In [ ]:
# Cell 0 — Imports & setup
import os
from pathlib import Path

import pandas as pd
import numpy as np
import duckdb
from difflib import get_close_matches, SequenceMatcher
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

load_dotenv()  # optional

def _norm(s: str) -> str:
    if s is None:
        return ""
    return (
        str(s)
        .strip()
        .lower()
        .replace("-", " ")
        .replace(".", " ")
        .replace(",", " ")
        .replace("  ", " ")
    )

def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def _mode_or_first(s: pd.Series):
    s = s.dropna()
    if s.empty:
        return None
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]

In [ ]:
# Cell 1 — Connect & inspect DB (DuckDB only; no fallbacks)
DB_PATH = "../data/mydb2024-25.duckdb"
con = duckdb.connect(database=DB_PATH, read_only=False)

print("Tables in the database:")
for (table_name,) in con.execute("SHOW TABLES").fetchall():
    columns = con.execute(f"DESCRIBE {table_name}").df()['column_name'].tolist()
    print(f"{table_name}: {columns}")

In [ ]:
# Cell 2 — fixtures_enhanced joined with fixtures to bring session_id
df_fixtures = con.execute("""
    SELECT 
        fe.*, 
        f.session_id
    FROM fixtures_enhanced AS fe
    LEFT JOIN fixtures AS f
      ON fe.fixtureId = f.fixtureId
""").df()

print("SPORTRADAR: Fixtures data (with session_id):")
display(df_fixtures.head())

In [ ]:
df_kx_all = con.execute(
    """
    SELECT 
        "full name"  AS full_name_kinexon, 
        "group name" AS group_name_kinexon, 
        "league id"  AS league_id, 
        session_id, 
        fixture_id
    FROM kinexon_positions
    WHERE "full name" IS NOT NULL 
      AND "league id" IS NOT NULL
      AND "group name" IS NOT NULL
    """
).df()

# enforce strings for league_id / group_name to avoid dtype mismatches
df_kx_all["league_id"]          = df_kx_all["league_id"].astype(str)
df_kx_all["group_name_kinexon"] = df_kx_all["group_name_kinexon"].astype(str)

# ambiguity diagnostics: how many distinct league_ids per name
kx_ids_per_name = (
    df_kx_all.groupby("full_name_kinexon")["league_id"]
             .nunique()
             .rename("unique_league_ids_per_name")
             .reset_index()
)

# choose mode league_id and mode group_name per Kinexon full name
kx_per_name = (
    df_kx_all.groupby("full_name_kinexon")
             .agg({
                 "league_id":          _mode_or_first,
                 "group_name_kinexon": _mode_or_first
             })
             .reset_index()
             .rename(columns={"league_id": "kin_league_id"})
)

# attach diagnostics
kx_per_name = kx_per_name.merge(kx_ids_per_name, on="full_name_kinexon", how="left")

# normalized keys for matching
kx_per_name["norm_key"]      = kx_per_name["full_name_kinexon"].map(_norm)
kx_per_name["kin_group_norm"] = kx_per_name["group_name_kinexon"].map(_norm)

print(f"KINEXON: distinct player names: {len(kx_per_name)}")
display(kx_per_name.head())

In [ ]:
df_sr_players = con.execute("""
    SELECT DISTINCT 
        p.personId, 
        p.nameFullLocal, 
        p.nameFullLatin, 
        p.teamName
    FROM players AS p
    JOIN match_events AS me 
      ON p.personId = me.personId
""").df()

# normalized fields (LOCAL name only + team)
df_sr_players["nameFullLocal"] = df_sr_players["nameFullLocal"].astype(str)
df_sr_players["teamName"]      = df_sr_players["teamName"].astype(str)
df_sr_players["name_local_norm"] = df_sr_players["nameFullLocal"].map(_norm)
df_sr_players["team_norm"]       = df_sr_players["teamName"].map(_norm)

print(f"SPORTRADAR: ALL players referenced in ALL match_events: {len(df_sr_players)}")
display(df_sr_players.head())

In [ ]:

# --- Cell 5 — Exact + fuzzy match ALL SR players to Kinexon per-name map (LOCAL ONLY) with TEAM CONSTRAINT
# Build maps (include team)
kin_map_key_to_name   = dict(zip(kx_per_name["norm_key"], kx_per_name["full_name_kinexon"]))
kin_map_key_to_league = dict(zip(kx_per_name["norm_key"], kx_per_name["kin_league_id"]))
kin_map_key_to_team   = dict(zip(kx_per_name["norm_key"], kx_per_name["kin_group_norm"]))
kin_keys = list(kin_map_key_to_name.keys())

rows_exact, rows_fuzzy = [], []

for _, r in df_sr_players.iterrows():
    pid          = r["personId"]
    sr_local     = r["nameFullLocal"]
    sr_team      = r["teamName"]
    key_local    = r["name_local_norm"]   # LOCAL only (no fallback to Latin)
    sr_team_norm = r["team_norm"]

    matched_key = None

    # --- Exact: name AND team must match
    if key_local in kin_keys and kin_map_key_to_team.get(key_local) == sr_team_norm:
        matched_key = key_local

    if matched_key:
        rows_exact.append((
            pid,
            sr_local,
            sr_team,
            kin_map_key_to_name[matched_key],
            kin_map_key_to_league[matched_key],
            kin_map_key_to_team[matched_key],  # kin team (normalized)
            1.0
        ))
        continue

    # --- Fuzzy: restrict candidate search to SAME TEAM only; LOCAL name only
    # Build eligible Kinexon keys that belong to the same team
    eligible_keys = [kk for kk in kin_keys if kin_map_key_to_team.get(kk) == sr_team_norm]

    # If there are no candidates in this team, skip (do not cross teams)
    if eligible_keys:
        cand = get_close_matches(key_local, eligible_keys, n=1, cutoff=0.84)
        if cand:
            best_key = cand[0]
            score = _similarity(key_local, best_key)
            rows_fuzzy.append((
                pid,
                sr_local,
                sr_team,
                kin_map_key_to_name[best_key],
                kin_map_key_to_league[best_key],
                kin_map_key_to_team[best_key],
                score
            ))

print(f"Exact matches: {len(rows_exact)} | Fuzzy matches: {len(rows_fuzzy)}")

# Fuzzy matches for inspection
df_fuzzy_matches = pd.DataFrame(
    rows_fuzzy,
    columns=[
        "personId",
        "nameFullLocal_sportradar",
        "teamName_sportradar",
        "full_name_kinexon",
        "league_id",
        "group_name_kinexon_norm",
        "similarity"
    ]
)
print("Fuzzy matches (for inspection):")
display(df_fuzzy_matches)

df_match_all = pd.DataFrame(
    rows_exact + rows_fuzzy,
    columns=[
        "personId",
        "nameFullLocal_sportradar",
        "teamName_sportradar",
        "full_name_kinexon",
        "league_id",
        "group_name_kinexon_norm",
        "similarity"
    ]
).drop_duplicates(subset=["personId"], keep="first")

print("Preview of per-player Kinexon IDs to write into players.league_id (team-aligned, local only):")
display(df_match_all.sort_values("similarity", ascending=False).head(30))

In [ ]:
# Enforce TEAM-ALIGNED updates (Kinexon "group name" == Sportradar "teamName")
# Assumes df_match_all contains:
#   ["personId", "nameFullLocal_sportradar", "teamName_sportradar",
#    "full_name_kinexon", "league_id", "group_name_kinexon_norm", "similarity"]

# 1) Build a SAFE mapping restricted to same-team matches (normalized)
df_map_safe = df_match_all.copy()
df_map_safe["team_norm_sportradar"] = df_map_safe["teamName_sportradar"].map(_norm)

# Keep only rows where normalized SR team equals normalized Kinexon group (team)
df_map_safe = df_map_safe.loc[
    df_map_safe["team_norm_sportradar"] == df_map_safe["group_name_kinexon_norm"],
    ["personId", "league_id", "teamName_sportradar", "team_norm_sportradar"]
].drop_duplicates(subset=["personId"], keep="first")

print(f"Team-aligned mappings to write: {len(df_map_safe)}")
if len(df_match_all) != len(df_map_safe):
    print(f"Skipped due to team mismatch: {len(df_match_all) - len(df_map_safe)}")

# 2) Reset league_id column (fresh write)
con.execute("""ALTER TABLE players DROP COLUMN IF EXISTS league_id""")
con.execute("""ALTER TABLE players ADD COLUMN IF NOT EXISTS league_id TEXT""")

# 3) Register temp table and UPDATE with an additional TEAM guard at SQL level too
#    (joins on personId AND exact teamName to be extra safe)
con.register("tmp_player_league_map_all", df_map_safe)

con.execute("""
    UPDATE players AS p
    SET league_id = tlm.league_id
    FROM tmp_player_league_map_all AS tlm
    WHERE p.personId = tlm.personId
      AND p.teamName = tlm.teamName_sportradar
""")

con.unregister("tmp_player_league_map_all")
print(f"✅ Updated players.league_id for {len(df_map_safe)} players (global, team-aligned).")

# 4) Diagnostics: show players updated and verify team alignment again
df_players_updated = con.execute("""
    SELECT *
    FROM players
    WHERE league_id IS NOT NULL
    ORDER BY teamName, nameFullLocal
""").df()
print("Players with updated league_id (team-aligned):")
display(df_players_updated)
1
# print of how many unique players in kinexon_positions have been matched to sportradar players
num_unique_players_matched = df_map_safe['personId'].nunique()
print(f"Number of unique Sportradar players matched to Kinexon data: {num_unique_players_matched}")

